*Notebook Last Validated: 2026-09-22*

Project UID (Internal, Prod): 1a1d6265-5eca-4840-a5d6-5c95ddd7ee62

# Rhino SDK - Runtime External Files

In this notebook we'll explain how to make files from a workgroup's S3 bucket available inside your code when it runs on FCP.

This is useful for reference files, model weights/checkpoints, or other artifacts your code needs that don't fit the shape of a registered Dataset (or that you'd rather not embed directly in the Code Object itself).

IMPORTANT: do NOT store sensitive information in run-time files!!!

For your code to be able to access files while running on your on-prem agent, there are 4 steps:
  1. Create an S3 bucket for the workgroup's files.
  2. Upload the relevant files to the bucket.
  3. Reference the files in your code.
  4. Tell the code run what files to download from the bucket.

## 1. Creating the bucket

The S3 code artifacts bucket is created by Rhino (on client request) as part of the onboarding process - you don't create or manage this bucket yourself.

If you don't already have one and would like one created, contact us at support@rhinofcp.com.

## 2. Uploading files to the bucket

There are many different ways to [upload files to an AWS S3 bucket](https://docs.aws.amazon.com/AmazonS3/latest/userguide/upload-objects.html).

For convenience, Rhino has created the following [script](https://github.com/RhinoHealth/user-resources/blob/main/utils/upload-file-to-s3.sh).

To use it, first set your AWS S3 credentials as environment variables in your terminal (these can be found on the FCP UI under **Settings -> Containers & Artifacts -> Container Registry Access**):

```bash
export AWS_ACCESS_KEY_ID=<ENTER_YOUR_ID>
export AWS_SECRET_ACCESS_KEY=<ENTER_YOUR_KEY>
```

Then, from the `utils/` directory this script is located in, call it with 4 arguments:

```bash
./upload-file-to-s3.sh <upload_directory> <storage_bucket> <bucket_prefix> <path_in_bucket>
```

- `<upload_directory>` - the local folder whose contents you want to upload.
- `<storage_bucket>` - the name of your workgroup's S3 bucket (see step 1).
- `<bucket_prefix>` - your workgroup's fixed prefix inside that bucket (provided to you during onboarding - this is *not* something you choose).
- `<path_in_bucket>` - the subfolder, under your prefix, to upload into. **This is the path you'll reference in step 3/4 below** (not the full bucket path, and not including `<bucket_prefix>`).

For example, uploading a local folder `./original_location` into a subfolder called `my_files`:

```bash
./upload-file-to-s3.sh ./original_location <storage_bucket> <bucket_prefix> my_files
```

## 3. Reference the files in your code

Once uploaded, your files show up inside the running container's **`/external_data`** folder, mirroring the folder structure *under your workgroup's prefix* (`<bucket_prefix>` from step 2 - you never include that prefix yourself when referencing a path; FCP scopes `/external_data` to it automatically).

So if you uploaded a file to `<path_in_bucket>` = `my_files` (i.e. it's at `s3://<storage_bucket>/<bucket_prefix>/my_files/model_params.txt`), inside the running container it's available at:

```
/external_data/my_files/model_params.txt
```

and you'd reference it in step 4 below as `"my_files/model_params.txt"`.

## 4. Use the files in the specific run

The rest of this notebook is a fully runnable, end-to-end example: it authenticates, connects to an existing project, picks a dataset (required by the API even though this example's code doesn't actually touch the dataset - it's only here to read the external file), then creates and runs a Code Object that reads and prints the contents of a runtime external file, and finally fetches and prints the run's actual output so you can see it worked.

### 4a. Setup

In [1]:
import os
from getpass import getpass
from textwrap import dedent

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeObjectRunInput,
    CodeTypes,
)
from rhino_health.lib.endpoints.endpoint import NameFilterMode

print("Setup Complete")

Setup Complete


### 4b. Authentication

Log in to the Rhino FCP. When prompted, provide your password.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [ ]:
USERNAME = "<YOUR_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

print("Logging in...")
session = rh.login(username=USERNAME, password=getpass())
user = session.current_user
print("Logged in")

Logging in...
Logged in


### 4c. Connecting to a project

Set `PROJECT` to the exact name of an existing project you want to run this in.

**Edge cases handled below:** no project with that name raises a clear error; multiple projects with that name print a warning naming which one (the most recently created) got used, since `get_project_by_name` would otherwise pick one silently.

In [4]:
PROJECT = "[User-Example] DO-456 Test"  # REPLACE WITH AN EXISTING PROJECT NAME

matches = session.project.search_for_projects_by_name(PROJECT, NameFilterMode.EXACT)
if not matches:
    raise ValueError(
        f"No project found named '{PROJECT}'. Double-check the spelling and capitalization "
        f"(matching is case-sensitive and exact), and that your account has access to it."
    )
if len(matches) > 1:
    newest = max(matches, key=lambda p: p.created_at)
    print(
        f"WARNING: {len(matches)} projects are named '{PROJECT}'. Using the most recently "
        f"created one (uid={newest.uid}, created {newest.created_at})."
    )

project = session.project.get_project_by_name(PROJECT)
print(f"Connected to project: {project.name} ({project.uid})")

Connected to project: [User-Example] DO-456 Test (1a1d6265-5eca-4840-a5d6-5c95ddd7ee62)


### 4d. Picking a dataset

Every Code Object run needs at least one input dataset (`input_dataset_uids` is required by the API), even though this example's code never actually reads from it - it only reads the runtime external file. Any dataset already in your project works. Set `DATASET_SEARCH` to a string that appears in the name of the dataset you want to use.

In [5]:
DATASET_SEARCH = "Dummy Dataset"  # REPLACE WITH AN EXISTING DATASET NAME IN THE PROJECT

datasets = project.search_for_datasets_by_name(DATASET_SEARCH, name_filter_mode=NameFilterMode.CONTAINS)
if not datasets:
    raise ValueError(
        f"No dataset found in project '{project.name}' matching '{DATASET_SEARCH}'. "
        f"Register or pick a different existing dataset in this project first."
    )
dataset = datasets[0]
print(f"Using dataset: {dataset.name} ({dataset.uid})")

Using dataset: Dummy Dataset (4891ea02-2bbc-4936-b950-091b591507cf)


### 4e. Creating & Running the Code Object

`EXTERNAL_FILE_PATH` is the path *relative to your workgroup's external files prefix* - matching step 2/3 above, a file uploaded to `.../DO-456/DO-456.txt` in the bucket is referenced here as `"DO-456/DO-456.txt"`, and shows up inside the running container at `/external_data/DO-456/DO-456.txt`.

Note: `python_version="3.9"` and this exact `requirements` list are kept as-is from a verified-working configuration - this specific combination is what routes the request through FCP's fast, simple synchronous Python execution path rather than a slower auto-built container. Changing them isn't necessarily broken, just unverified here.

In [ ]:
EXTERNAL_FILE_PATH = "DO-456/DO-456.txt" # REPLACE WITH THE PATH OF AN EXISTING FILE IN THE PROJECT'S RUNTIME EXTERNAL FILES
CODE_OBJECT_NAME = "Print Runtime External File"

code_object_input = CodeObjectCreateInput(
    name=CODE_OBJECT_NAME,
    description="Reads and prints the contents of a runtime external file",
    code_type=CodeTypes.PYTHON_CODE,
    project_uid=project.uid,
    config={
        "python_version": "3.9",
        "requirements": ["numpy == 1.22.*", "pandas ~= 1.4.2"],
        "python_code": dedent(f"""
            from pathlib import Path
            text = Path('/external_data/{EXTERNAL_FILE_PATH}').read_text()
            print(text)
            """),
        "code_execution_mode": "snippet",
    },
    input_data_schema_uids=[None],
    output_data_schema_uids=[None],
)
# return_existing=False + add_version_if_exists=True: if a Code Object named CODE_OBJECT_NAME
# already exists (e.g. from a prior run), create a NEW VERSION instead of silently returning
# the old one - otherwise changing EXTERNAL_FILE_PATH above would never take effect.
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
print(f"Created code object: {code_object.name} ({code_object.uid})")

run_params = CodeObjectRunInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[[dataset.uid]],
    output_dataset_naming_templates=["{{input_dataset_names.0}} - file contents"],
    external_storage_file_paths=[EXTERNAL_FILE_PATH],
    timeout_seconds=600,
    sync=True,
)
code_run = session.code_object.run_code_object(run_params)
code_run = code_run.wait_for_completion(600)
print(f"Run finished with status: {code_run.status}")

Created code object: Print Runtime External File (2992d641-4951-47db-a266-620e6c7b637c)
Run finished with status: CodeRunStatus.COMPLETED


#### 4f. Viewing output

In the FCP UI, go to Code Runs, and then click on the completed code run.

Assuming you have permission to view logs, this is where you'll see the contents of `DO-456/DO-456.txt` printed by the code object.

## Getting Help

For additional support, check out [RhinoDocs](https://docs.rhinofcp.com/) or reach out to [support@rhinofcp.com](mailto:support@rhinofcp.com).